# Document-level context (FLERT), window 32 — deberta `{1e-5, 5}`

Same config as E08 with one change: `--context-window 32`. 32 dataset tokens per
side, taken from the sentence's own document, attended (mask 1) and never scored
(label `-100`).

**Why 32 and not more.** `results/context_fill_rate.md`: at 32 per side, 64% of
htfl sentences get a full window against 96-99% for the training domains. At 128
htfl falls to 14%, at 210 to 2%. Widening the window widens the train/test
mismatch, so 32 is measured first and 64 only if 32 helps.

**The baseline is E08**, not E05 — same five seeds, same code path, and E08 is the
run that already carries `span_f1` and the invalid-tag counts.

| | E08 baseline |
|---|--:|
| equi | 0.5592 +/- 0.0090 |
| htfl | 0.5782 +/- 0.0230 |

Report equi and htfl **separately, never pooled** — they have different fill rates
and that is the point of the experiment.

Budget ~75 min: context roughly doubles eval time and lengthens training, against
E05's 440 s per run. Sidebar: **GPU T4 x2**, **Internet On**.


In [ ]:
# 1. Kaggle guard, internet, GPU, and the command helper.
import os, sys, socket, subprocess, pathlib, shutil, json

if not pathlib.Path('/kaggle').is_dir():
    raise SystemExit('This notebook is for Kaggle.')
try:
    socket.create_connection(('github.com', 443), timeout=10).close()
except OSError as e:
    raise SystemExit(f'No internet ({e}). Sidebar -> Session options -> Internet -> On.')

import torch
assert torch.cuda.is_available(), 'No GPU. Sidebar -> Session options -> Accelerator -> GPU.'
print(torch.cuda.get_device_name(0), '|', torch.__version__, '| cuda', torch.version.cuda)

WORK = pathlib.Path('/kaggle/working')
REPO = pathlib.Path('/tmp/ate-acter')
WINDOW = 32
GROUP = f'context/deberta_ctx{WINDOW}'
SRC = REPO / 'results/runs' / GROUP

def run(*args, cwd=None):
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=cwd, env=env)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {" ".join(str(a) for a in args)}')

In [ ]:
# 2. Clone the project and the corpus into /tmp.
shutil.rmtree(REPO, ignore_errors=True)
run('git', 'clone', '-q', 'https://github.com/ahmedwaleedaref/ATE-ACTER.git', REPO)
run('git', 'clone', '-q', 'https://github.com/AylaRT/ACTER.git', REPO / 'data/raw/ACTER')
run('git', 'checkout', '-q', 'f05b09e985cad37eeaa8daa8b3f383197aa5324e',
    cwd=REPO / 'data/raw/ACTER')

# Tripwires read the SOURCE. Without the FLERT commit this notebook runs the
# sentence-level baseline for 75 minutes and reports it as a context result.
for path, needle, why in (
    ('src/models/run_train.py', '--context-window',      'the flag is absent; context would be 0'),
    ('src/data/dataset.py',     'context_window: int = 0', 'build_examples cannot take a window'),
    ('src/data/dataset.py',     'recover_sentence_labels', 'recovery would not offset past the context'),
    ('src/models/run_train.py', '--dump-terms',          'no term list for the breakdown'),
):
    assert needle in (REPO / path).read_text(), f'clone predates {needle!r} in {path}: {why}'
assert (REPO / 'data/raw/ACTER/en/htfl/annotated').is_dir(), 'ACTER checkout looks wrong'
print(subprocess.run(['git','log','--oneline','-1'], cwd=REPO,
                     capture_output=True, text=True).stdout)

In [ ]:
# 3. Pinned installs.
run(sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==5.16.1', 'tokenizers==0.23.1', 'safetensors==0.8.0',
    'huggingface_hub==1.29.0', 'sentencepiece==0.2.2', 'protobuf==7.36.0',
    'PyYAML==6.0.3', 'pytest==8.3.2')

def _pip(*a):
    return subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a],
                          capture_output=True, text=True).returncode == 0
HAVE_SEQEVAL = (_pip('--no-build-isolation', 'seqeval==1.2.2')
                or (_pip('setuptools<81', 'wheel')
                    and _pip('--no-build-isolation', 'seqeval==1.2.2')))
print('seqeval:', 'installed' if HAVE_SEQEVAL else 'UNAVAILABLE -- training unaffected')

In [ ]:
# 4. Tests, then THE GATE. The gate is what makes a context result readable:
#    context positions must be -100, so the positive rate over scored positions
#    must be byte-identical to the sentence-level rate. If a context label reaches
#    the loss, the model trains on labels it will not be scored on and the whole
#    comparison is meaningless -- with nothing crashing.
import importlib
for m in ('torch', 'transformers', 'tokenizers', 'numpy'):
    print(f'{m:14} {importlib.import_module(m).__version__}')
skip = [] if HAVE_SEQEVAL else ['--ignore=tests/test_seqeval_agreement.py']
run(sys.executable, '-m', 'pytest', 'tests/', '-q', *skip, cwd=REPO)

gate = f'''
import dataclasses
from src.data.dataset import (ATEDataset, build_dataloader, build_examples,
                              get_tokenizer, load_train_config)
from src.data.align import positive_rate, IGNORE_INDEX
from src.stats.loading import load_config
cfg = dataclasses.replace(load_train_config(), model_name="microsoft/deberta-v3-base")
tok, dcfg = get_tokenizer(cfg), load_config()
for dom, filt in (("corp", None), ("equi", None), ("wind", 2), ("htfl", None)):
    rates = {{}}
    for W in (0, {WINDOW}):
        ex = build_examples(dom, tokenizer=tok, truncation=True, max_length=cfg.max_length,
                            filter_max_tokens=filt, context_window=W, data_cfg=dcfg)
        ld = build_dataloader(ATEDataset(ex), tokenizer=tok, batch_size=32,
                              shuffle=False, length_grouped=False)
        rates[W] = positive_rate(ld)[0]
    leaks = sum(1 for e in ex for w, l in zip(e.word_ids, e.labels)
                if w is not None and not (e.n_left <= w < e.n_left + len(e.tokens))
                and l != IGNORE_INDEX)
    assert abs(rates[0] - rates[{WINDOW}]) < 1e-12, f"{{dom}}: rate moved {{rates}}"
    assert leaks == 0, f"{{dom}}: {{leaks}} context labels reached the loss"
    print(f"  {{dom:6}} rate {{rates[0]:.6f}} == {{rates[{WINDOW}]:.6f}}, 0 leaks")
print("GATE PASSED")
'''
print('\ncontext gate:')
run(sys.executable, '-c', gate, cwd=REPO)

In [ ]:
# 5. Seed 42 alone. Inspect before spending an hour on the rest.
def train(seed):
    run(sys.executable, '-m', 'src.models.run_train',
        '--model', 'microsoft/deberta-v3-base',
        '--lr', '1e-5', '--epochs', 5, '--context-window', WINDOW,
        '--group', GROUP, '--seed', seed, '--dump-terms',
        '--reason', f'FLERT document context, window {WINDOW}, vs E08 baseline',
        cwd=REPO)
    r = json.loads((SRC / f'seed_{seed}.json').read_text())
    t = r['test']
    assert r['config']['context_window'] == WINDOW, 'context_window did not reach the run'
    assert r['test_term_list']['n_terms'] == t['n_pred_types']
    print(f"    SEED {seed}: best_epoch={r['best_epoch']} equi={r['best_equi_f1']:.4f} "
          f"htfl={t['list_ann_f1']:.4f} span_f1={t['span_f1']:.4f} types={t['n_pred_types']}")
    print(f"      vs E08 baseline: equi 0.5592 +/- 0.0090, htfl 0.5782 +/- 0.0230 "
          f"| {r['wall_time_sec']}s")
    return r

def persist():
    dest = WORK / GROUP
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(SRC, dest)
    print(f'  -> copied to {dest}')

print(f'########## seed 42, context window {WINDOW} ##########')
train(42)
persist()

In [ ]:
# 6. Seeds 43-46. Output re-copied after each, so a dropped session costs one run.
for s in (43, 44, 45, 46):
    print(f'\n########## seed {s} ##########')
    train(s)
    persist()

In [ ]:
# 7. Five-seed summary against the E08 baseline, paired on the seed.
import statistics
runs = {json.loads(p.read_text())['seed']: json.loads(p.read_text())
        for p in sorted((WORK / GROUP).glob('seed_*.json'))}
if len(runs) == 5:
    E08 = {'equi': {42: 0.5673, 43: 0.5442, 44: 0.5581, 45: 0.5627, 46: 0.5639},
           'htfl': {42: 0.6046, 43: 0.5783, 44: 0.5652, 45: 0.5474, 46: 0.5953}}
    for domain, get in (('equi', lambda r: r['best_equi_f1']),
                        ('htfl', lambda r: r['test']['list_ann_f1'])):
        new = [get(runs[s]) for s in (42, 43, 44, 45, 46)]
        diff = [get(runs[s]) - E08[domain][s] for s in (42, 43, 44, 45, 46)]
        m, sd = statistics.mean(diff), statistics.stdev(diff)
        t = m / (sd / len(diff) ** 0.5) if sd else float('inf')
        print(f'{domain}: {statistics.mean(new):.4f} +/- {statistics.stdev(new):.4f}  '
              f'paired diff {m:+.4f} (s_d {sd:.4f}, t {t:.2f}, '
              f'{sum(d > 0 for d in diff)}+/{sum(d < 0 for d in diff)}-)')
    print('\ndf=4: 2.132 suggestive, 2.776 take seriously. Report equi and htfl '
          'SEPARATELY -- they have different fill rates, which is the experiment.')
else:
    print(f'{len(runs)} of 5 seeds -- rerun cell 6.')
for p in sorted((WORK / 'context').rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(WORK)}  {p.stat().st_size/1024:.0f} KB')